# Agentic RAG Orchestration for SEC Edgar

This notebook implements a multi-agent RAG system for querying SEC Edgar documents with:
- **Planner Agent**: Decomposes complex questions into sub-questions
- **Retrieval Agents**: Fetches relevant SEC Edgar documents
- **Verification Agent**: Self-RAG style verification of relevance and correctness
- **Synthesis Agent**: Combines results with proper citations


In [ ]:
# Install required packages
!pip install uv
!uv pip install -q langchain langchain-core langchain_text_splitters langchain-openai langchain-community langgraph sec-edgar-downloader chromadb langchain-chroma pypdf tiktoken sec2md edgartools


In [ ]:
import os
from typing import List, Dict, Any, Optional, TypedDict
from datetime import datetime
import json

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_classic.chains import RetrievalQA
from langchain_classic.callbacks import StdOutCallbackHandler

import chromadb
from chromadb.config import Settings

import glob
import sec2md
from edgar import Company, set_identity

# Set your OpenAI API key
os.environ["OPENAI_API_KEY"] = "YOUR-APIKEY-HERE"  # Replace with your actual key

# Initialize LLM and embeddings
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings()


## 1. Planner Agent - Question Decomposition


In [ ]:
class PlannerAgent:
    """Decomposes complex questions into sub-questions for better retrieval."""

    def __init__(self, llm):
        self.llm = llm
        self.prompt_template = ChatPromptTemplate.from_messages([
            ("system", """You are a planning agent that decomposes complex questions about SEC Edgar filings into
            smaller, more focused sub-questions. Each sub-question should be:
            1. Specific and answerable
            2. Focused on a single aspect of the original question
            3. Suitable for document retrieval

            Return a JSON list of sub-questions."""),
            ("human", "Original question: {question}\n\nDecompose this into sub-questions. Return only a JSON array of strings.")
        ])

    def decompose(self, question: str) -> List[str]:
        """Decompose a question into sub-questions."""
        chain = self.prompt_template | self.llm
        response = chain.invoke({"question": question})

        try:
            # Extract JSON from response
            content = response.content.strip()
            if content.startswith("```json"):
                content = content[7:]
            if content.startswith("```"):
                content = content[3:]
            if content.endswith("```"):
                content = content[:-3]
            content = content.strip()

            sub_questions = json.loads(content)
            if isinstance(sub_questions, list):
                return sub_questions
            else:
                return [question]  # Fallback to original question
        except:
            # Fallback: return original question
            return [question]

    def plan(self, question: str) -> Dict[str, Any]:
        """Create a plan with sub-questions."""
        sub_questions = self.decompose(question)
        return {
            "original_question": question,
            "sub_questions": sub_questions,
            "num_sub_questions": len(sub_questions)
        }

# Initialize planner
planner = PlannerAgent(llm)


In [ ]:
# Test planner
test_question = "What are the main risks and financial performance metrics for Apple Inc. in their latest 10-K filing?"
plan = planner.plan(test_question)
print("Original Question:", plan["original_question"])
print("\nSub-questions:")
for i, sq in enumerate(plan["sub_questions"], 1):
    print(f"{i}. {sq}")


Original Question: What are the main risks and financial performance metrics for Apple Inc. in their latest 10-K filing?

Sub-questions:
1. What are the main risks disclosed by Apple Inc. in their latest 10-K filing?
2. What financial performance metrics are reported by Apple Inc. in their latest 10-K filing?
3. How does Apple Inc. define its risk factors in the latest 10-K filing?
4. What specific financial performance metrics does Apple Inc. highlight in the latest 10-K filing?
5. Are there any changes in risk factors compared to the previous 10-K filing for Apple Inc.?
6. What revenue figures are reported by Apple Inc. in their latest 10-K filing?
7. What net income figures are reported by Apple Inc. in their latest 10-K filing?
8. What is the trend in Apple's financial performance metrics over the last few years as per the latest 10-K filing?
9. What operational risks does Apple Inc. mention in their latest 10-K filing?
10. What market risks are identified by Apple Inc. in their la

## 2. Retrieval Agents - SEC Edgar Document Retrieval


In [ ]:
class SECEdgarRetrievalAgent:
    """Retrieves and processes SEC Edgar documents."""

    def __init__(self, llm, embeddings, company_ticker: str, filing_type: str = "10-K"):
        self.llm = llm
        self.embeddings = embeddings
        self.company_ticker = company_ticker.upper()
        self.filing_type = filing_type
        self.vectorstore = None
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200,
            length_function=len,
        )
        self.loaded_documents: List[Document] = [] # Store loaded documents
        set_identity("social@testopia.io")

    def download_filings(self, num_filings: int = 1):
        """Download SEC Edgar filings for the company."""
        print(f"Downloading {self.filing_type} filings for {self.company_ticker}...")
        try:
            company = Company(self.company_ticker)
            filing = company.get_filings(form=self.filing_type).latest()

            print(f"Downloaded {self.filing_type} filings for {self.company_ticker}")

            return filing
        except Exception as e:
            print(f"Error downloading filings: {e}")
            # Do not return mock documents here, let process_documents handle the fallback

    def _create_mock_documents(self) -> List[Document]:
        """Create mock documents for demonstration if download fails or no files found."""
        return [
            Document(
                page_content=f"Mock {self.filing_type} filing content for {self.company_ticker}. "
                           f"This is a placeholder document. In production, this would contain "
                           f"the actual SEC Edgar filing content.",
                metadata={"source": f"{self.company_ticker}_{self.filing_type}_mock.pdf",
                         "ticker": self.company_ticker,
                         "filing_type": self.filing_type}
            )
        ]

    def process_documents(self, documents: Optional[List[Document]] = None) -> Chroma:
        """Process documents and create vector store."""
        if documents is None:
            # Attempt to load from downloaded files if available
            if not self.loaded_documents: # Only load if not already loaded
                filing = self.download_filings()

                md = sec2md.convert_to_markdown(filing.html(), return_pages=True)
                sections = sec2md.extract_sections(md, filing_type=self.filing_type)

                self.loaded_documents = [
                      Document(
                          page_content=sec.markdown(),
                          metadata={
                              "source": f"{sec.item} - {sec.item_title} - pages{str(sec.page_range)}",
                              "ticker": self.company_ticker,
                              "filing_type": self.filing_type,
                          }
                      )
                      for sec in sections
                  ]
            documents = self.loaded_documents

            # Fallback to mock documents if _load_documents_from_downloaded_files also failed to find/load any
            if not documents:
                documents = self._create_mock_documents()

        # Split documents
        texts = self.text_splitter.split_documents(documents)

        # Create vector store
        self.vectorstore = Chroma.from_documents(
            documents=texts,
            embedding=self.embeddings,
            collection_name=f"{self.company_ticker}_{self.filing_type}"
        )

        return self.vectorstore

    def retrieve(self, query: str, k: int = 5) -> List[Document]:
        """Retrieve relevant documents for a query."""
        if self.vectorstore is None:
            print("Vectorstore not initialized. Processing documents first.")
            self.process_documents()

        retriever = self.vectorstore.as_retriever(search_kwargs={"k": k})
        docs = retriever.invoke(query)

        return docs

    def retrieve_with_scores(self, query: str, k: int = 5) -> List[Dict[str, Any]]:
        """Retrieve documents with relevance scores."""
        if self.vectorstore is None:
            print("Vectorstore not initialized. Processing documents first.")
            self.process_documents()

        # Use similarity search with scores
        docs_with_scores = self.vectorstore.similarity_search_with_score(query, k=k)

        results = []
        for doc, score in docs_with_scores:
            results.append({
                "document": doc,
                "score": score,
                "content": doc.page_content,
                "metadata": doc.metadata
            })

        return results

# Example usage
retrieval_agent = SECEdgarRetrievalAgent(llm, embeddings, company_ticker="AAPL", filing_type="10-K")
retrieval_agent.process_documents()

Downloaded 10-K filings for AAPL


In [ ]:
# Test retrieval
test_query = "What are the main financial risks?"
results = retrieval_agent.retrieve_with_scores(test_query, k=3)
print(f"Retrieved {len(results)} documents:")
for i, result in enumerate(results, 1):
    print(f"\n--- Result {i} (Score: {result['score']:.4f}) ---")
    print(f"Content: {result['content'][:200]}...")
    print(f"Metadata: {result['metadata']}")


Retrieved 3 documents:

--- Result 1 (Score: 0.3353) ---
Content: *The Company is exposed to credit risk and fluctuations in the values of its investment portfolio.*

The Company’s investments can be negatively affected by changes in liquidity, credit deterioration,...
Metadata: {'filing_type': '10-K', 'source': 'ITEM 1A - Risk Factors - pages(7, 19)', 'ticker': 'AAPL'}

--- Result 2 (Score: 0.3353) ---
Content: *The Company is exposed to credit risk and fluctuations in the values of its investment portfolio.*

The Company’s investments can be negatively affected by changes in liquidity, credit deterioration,...
Metadata: {'ticker': 'AAPL', 'filing_type': '10-K', 'source': 'ITEM 1A - Risk Factors - pages(7, 19)'}

--- Result 3 (Score: 0.3409) ---
Content: If the Company is unable to compete successfully, its business, reputation, results of operations, financial condition and stock price can be materially adversely affected.

Business Risks

*To remain...
Metadata: {'source': 'ITEM 1A 

## 3. Verification Agent - Self-RAG Style


In [ ]:
class VerificationAgent:
    """Self-RAG style verification agent that checks relevance and correctness."""

    def __init__(self, llm):
        self.llm = llm
        self.relevance_prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a verification agent that evaluates the relevance of retrieved documents
            to a given question. Rate each document on a scale of 0.0 to 1.0 where:
            - 1.0 = Highly relevant and directly answers the question
            - 0.5 = Somewhat relevant but may not fully answer
            - 0.0 = Not relevant

            Return a JSON object with relevance scores for each document."""),
            ("human", """Question: {question}

            Document: {document_content}

            Rate the relevance of this document to the question. Return only a JSON object with:
            {{
                "relevance_score": <float 0.0-1.0>,
                "reasoning": "<brief explanation>",
                "is_relevant": <boolean>
            }}""")
        ])

        self.correctness_prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a verification agent that checks if an answer is correct and well-supported
            by the provided documents. Evaluate:
            1. Factual correctness
            2. Completeness
            3. Support from documents

            Return a JSON object with your evaluation."""),
            ("human", """Question: {question}

            Answer: {answer}

            Supporting Documents:
            {documents}

            Evaluate the correctness and support. Return only a JSON object with:
            {{
                "correctness_score": <float 0.0-1.0>,
                "is_supported": <boolean>,
                "missing_information": [<list of missing info>],
                "reasoning": "<brief explanation>"
            }}""")
        ])

    def verify_relevance(self, question: str, documents: List[Document]) -> List[Dict[str, Any]]:
        """Verify relevance of documents to the question."""
        verified_docs = []

        for doc in documents:
            chain = self.relevance_prompt | self.llm
            response = chain.invoke({
                "question": question,
                "document_content": doc.page_content[:2000]  # Limit content length
            })

            try:
                content = response.content.strip()
                if content.startswith("```json"):
                    content = content[7:]
                if content.startswith("```"):
                    content = content[3:]
                if content.endswith("```"):
                    content = content[:-3]
                content = content.strip()

                verification = json.loads(content)
                verification["document"] = doc
                verified_docs.append(verification)
            except Exception as e:
                # Fallback: assume relevant
                verified_docs.append({
                    "document": doc,
                    "relevance_score": 0.5,
                    "reasoning": "Could not parse verification",
                    "is_relevant": True
                })

        # Sort by relevance score
        verified_docs.sort(key=lambda x: x.get("relevance_score", 0), reverse=True)
        return verified_docs

    def verify_correctness(self, question: str, answer: str, documents: List[Document]) -> Dict[str, Any]:
        """Verify correctness and support of an answer."""
        docs_text = "\n\n".join([
            f"Document {i+1}:\n{doc.page_content[:500]}"
            for i, doc in enumerate(documents)
        ])

        chain = self.correctness_prompt | self.llm
        response = chain.invoke({
            "question": question,
            "answer": answer,
            "documents": docs_text
        })

        try:
            content = response.content.strip()
            if content.startswith("```json"):
                content = content[7:]
            if content.startswith("```"):
                content = content[3:]
            if content.endswith("```"):
                content = content[:-3]
            content = content.strip()

            return json.loads(content)
        except Exception as e:
            return {
                "correctness_score": 0.5,
                "is_supported": True,
                "missing_information": [],
                "reasoning": "Could not parse verification"
            }

# Initialize verification agent
verification_agent = VerificationAgent(llm)


In [ ]:
# Test verification
test_docs = retrieval_agent.retrieve("financial risks", k=2)
verified = verification_agent.verify_relevance("What are the main financial risks?", test_docs)
print("Relevance Verification:")
for i, v in enumerate(verified, 1):
    print(f"\nDocument {i}:")
    print(f"  Relevance Score: {v.get('relevance_score', 0):.2f}")
    print(f"  Is Relevant: {v.get('is_relevant', False)}")
    print(f"  Reasoning: {v.get('reasoning', 'N/A')}")


Relevance Verification:

Document 1:
  Relevance Score: 1.00
  Is Relevant: True
  Reasoning: The document directly addresses financial risks, specifically mentioning credit risk and fluctuations in investment values, which are key components of financial risk.

Document 2:
  Relevance Score: 1.00
  Is Relevant: True
  Reasoning: The document directly addresses financial risks, specifically mentioning credit risk and fluctuations in investment values, which are key components of financial risk.


## 4. Synthesis Agent - Answer Generation with Citations


In [ ]:
class SynthesisAgent:
    """Synthesizes answers from verified documents with proper citations."""

    def __init__(self, llm):
        self.llm = llm
        self.synthesis_prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a synthesis agent that creates comprehensive answers from multiple
            verified documents. Your answers must:
            1. Be accurate and well-supported
            2. Include citations in the format [Document N] where N is the document number
            3. Synthesize information from multiple sources when relevant
            4. Clearly indicate when information is not available in the documents

            Format your response with:
            - A clear, direct answer
            - Supporting details with citations
            - A list of sources at the end"""),
            ("human", """Question: {question}

            Verified Documents:
            {documents}

            Create a comprehensive answer with citations.""")
        ])

    def synthesize(self, question: str, verified_documents: List[Dict[str, Any]]) -> Dict[str, Any]:
        """Synthesize answer from verified documents."""
        # Filter to only relevant documents
        relevant_docs = [
            v for v in verified_documents
            if v.get("is_relevant", True) and v.get("relevance_score", 0) > 0.3
        ]

        if not relevant_docs:
            return {
                "answer": "I could not find relevant information in the documents to answer this question.",
                "citations": [],
                "sources": [],
                "num_sources": 0
            }

        # Format documents for prompt
        docs_text = ""
        sources = []
        for i, v in enumerate(relevant_docs, 1):
            doc = v["document"]
            docs_text += f"\n\n[Document {i}]\n"
            docs_text += f"Content: {doc.page_content[:1500]}\n"
            docs_text += f"Relevance Score: {v.get('relevance_score', 0):.2f}\n"

            source_info = {
                "document_id": i,
                "metadata": doc.metadata,
                "relevance_score": v.get("relevance_score", 0),
                "reasoning": v.get("reasoning", "")
            }
            sources.append(source_info)

        # Generate answer
        chain = self.synthesis_prompt | self.llm
        response = chain.invoke({
            "question": question,
            "documents": docs_text
        })

        answer = response.content

        # Extract citations from answer
        citations = self._extract_citations(answer)

        return {
            "answer": answer,
            "citations": citations,
            "sources": sources,
            "num_sources": len(citations)
        }

    def _extract_citations(self, text: str) -> List[int]:
        """Extract document citations from text."""
        import re
        citations = re.findall(r'\[Document (\d+)\]', text)
        return [int(c) for c in citations]

# Initialize synthesis agent
synthesis_agent = SynthesisAgent(llm)


In [ ]:
# Test synthesis
test_verified = verification_agent.verify_relevance("What are the main financial risks?", test_docs)
synthesized = synthesis_agent.synthesize("What are the main financial risks?", test_verified)
print("Synthesized Answer:")
print(synthesized["answer"])
print(f"\nCitations: {synthesized['citations']}")
print(f"Number of sources: {synthesized['num_sources']}")


Synthesized Answer:
The main financial risks that companies typically face include credit risk, market risk, liquidity risk, and interest rate risk. 

1. **Credit Risk**: This is the risk of loss arising from a borrower’s failure to repay a loan or meet contractual obligations. Companies are particularly exposed to credit risk on trade accounts receivable, vendor non-trade receivables, and prepayments related to long-term supply agreements. This risk can increase during periods of economic downturn, as the likelihood of defaults rises [Document 1].

2. **Market Risk**: Companies are also exposed to fluctuations in the values of their investment portfolios. Changes in market conditions, such as economic downturns, political instability, or changes in liquidity, can negatively impact the value of investments. This can lead to significant losses that adversely affect a company's financial condition and stock price [Document 1][Document 2].

3. **Liquidity Risk**: This refers to the risk t

## 5. Agentic RAG Orchestration - Complete Workflow


In [ ]:
class AgenticRAGOrchestrator:
    """Orchestrates the complete Agentic RAG workflow."""

    def __init__(self, company_ticker: str, filing_type: str = "10-K"):
        self.company_ticker = company_ticker
        self.filing_type = filing_type

        # Initialize all agents
        self.planner = PlannerAgent(llm)
        self.retrieval_agent = SECEdgarRetrievalAgent(llm, embeddings, company_ticker, filing_type)
        self.verification_agent = VerificationAgent(llm)
        self.synthesis_agent = SynthesisAgent(llm)

        # Initialize retrieval agent's vector store
        self.retrieval_agent.process_documents()

    def query(self, question: str, verbose: bool = True) -> Dict[str, Any]:
        """Execute the complete Agentic RAG pipeline."""
        if verbose:
            print(f"🔍 Processing question: {question}\n")

        # Step 1: Planning - Decompose question
        if verbose:
            print("📋 Step 1: Planning - Decomposing question...")
        plan = self.planner.plan(question)
        sub_questions = plan["sub_questions"]
        if verbose:
            print(f"   Generated {len(sub_questions)} sub-question(s)\n")

        # Step 2: Retrieval - Get documents for each sub-question
        if verbose:
            print("🔎 Step 2: Retrieval - Fetching relevant documents...")
        all_retrieved_docs = []
        for i, sq in enumerate(sub_questions, 1):
            if verbose:
                print(f"   Sub-question {i}: {sq}")
            docs = self.retrieval_agent.retrieve(sq, k=5)
            all_retrieved_docs.extend(docs)
            if verbose:
                print(f"      Retrieved {len(docs)} document(s)")

        # Remove duplicates based on content
        seen = set()
        unique_docs = []
        for doc in all_retrieved_docs:
            content_hash = hash(doc.page_content[:100])
            if content_hash not in seen:
                seen.add(content_hash)
                unique_docs.append(doc)

        if verbose:
            print(f"   Total unique documents: {len(unique_docs)}\n")

        # Step 3: Verification - Verify relevance
        if verbose:
            print("✅ Step 3: Verification - Verifying document relevance...")
        verified_docs = self.verification_agent.verify_relevance(question, unique_docs)
        relevant_docs = [v for v in verified_docs if v.get("is_relevant", True) and v.get("relevance_score", 0) > 0.3]
        if verbose:
            print(f"   Verified {len(verified_docs)} document(s)")
            print(f"   {len(relevant_docs)} document(s) passed relevance threshold\n")

        # Step 4: Synthesis - Generate answer with citations
        if verbose:
            print("📝 Step 4: Synthesis - Generating answer with citations...")
        result = self.synthesis_agent.synthesize(question, verified_docs)

        # Step 5: Final verification - Check answer correctness
        if verbose:
            print("🔍 Step 5: Final Verification - Checking answer correctness...")
        correctness = self.verification_agent.verify_correctness(
            question,
            result["answer"],
            [v["document"] for v in relevant_docs]
        )
        result["correctness"] = correctness

        if verbose:
            print(f"   Correctness Score: {correctness.get('correctness_score', 0):.2f}")
            print(f"   Is Supported: {correctness.get('is_supported', False)}\n")

        # Compile final result
        final_result = {
            "question": question,
            "plan": plan,
            "retrieved_documents": len(unique_docs),
            "verified_documents": len(verified_docs),
            "relevant_documents": len(relevant_docs),
            "answer": result["answer"],
            "citations": result["citations"],
            "sources": result["sources"],
            "correctness": correctness,
            "metadata": {
                "company_ticker": self.company_ticker,
                "filing_type": self.filing_type,
                "timestamp": datetime.now().isoformat()
            }
        }

        return final_result

    def format_response(self, result: Dict[str, Any]) -> str:
        """Format the result for display."""
        output = []
        output.append("=" * 80)
        output.append("AGENTIC RAG RESPONSE")
        output.append("=" * 80)
        output.append(f"\nQuestion: {result['question']}")
        output.append(f"\nCompany: {result['metadata']['company_ticker']}")
        output.append(f"Filing Type: {result['metadata']['filing_type']}")
        output.append(f"\nProcessing Summary:")
        output.append(f"  - Sub-questions generated: {result['plan']['num_sub_questions']}")
        output.append(f"  - Documents retrieved: {result['retrieved_documents']}")
        output.append(f"  - Documents verified: {result['verified_documents']}")
        output.append(f"  - Relevant documents: {result['relevant_documents']}")
        output.append(f"  - Correctness score: {result['correctness'].get('correctness_score', 0):.2f}")
        output.append("\n" + "-" * 80)
        output.append("ANSWER:")
        output.append("-" * 80)
        output.append(result['answer'])
        output.append("\n" + "-" * 80)
        output.append("SOURCES:")
        output.append("-" * 80)
        for i, source in enumerate(result['sources'], 1):
            output.append(f"\nSource {i}:")
            output.append(f"  Relevance Score: {source['relevance_score']:.2f}")
            output.append(f"  Metadata: {source['metadata']}")
        output.append("\n" + "=" * 80)

        return "\n".join(output)

# Initialize orchestrator
orchestrator = AgenticRAGOrchestrator(company_ticker="AAPL", filing_type="10-K")


Downloaded 10-K filings for AAPL


In [ ]:
# Example query
question = "What are the main risks and financial performance metrics for Apple Inc.?"
result = orchestrator.query(question, verbose=True)
print("\n" + orchestrator.format_response(result))


🔍 Processing question: What are the main risks and financial performance metrics for Apple Inc.?

📋 Step 1: Planning - Decomposing question...
   Generated 10 sub-question(s)

🔎 Step 2: Retrieval - Fetching relevant documents...
   Sub-question 1: What are the main risks identified in Apple Inc.'s latest 10-K filing?
      Retrieved 5 document(s)
   Sub-question 2: What financial performance metrics are reported in Apple Inc.'s latest quarterly earnings report?
      Retrieved 5 document(s)
   Sub-question 3: How does Apple Inc. assess its market risk in its SEC filings?
      Retrieved 5 document(s)
   Sub-question 4: What are the key financial ratios reported by Apple Inc. in its most recent financial statements?
      Retrieved 5 document(s)
   Sub-question 5: What specific operational risks does Apple Inc. mention in its SEC disclosures?
      Retrieved 5 document(s)
   Sub-question 6: How has Apple Inc.'s revenue growth trended over the past few years according to its filings?
   

## 6. Interactive Query Interface


In [ ]:
def query_sec_edgar(question: str, company_ticker: str = "AAPL", filing_type: str = "10-K", verbose: bool = False):
    """
    Convenience function to query SEC Edgar documents using Agentic RAG.

    Args:
        question: The question to ask
        company_ticker: Company ticker symbol (e.g., "AAPL", "MSFT")
        filing_type: Type of SEC filing (e.g., "10-K", "10-Q", "8-K")
        verbose: Whether to show detailed processing steps

    Returns:
        Dictionary with answer, citations, and metadata
    """
    # Create orchestrator for the specific company
    orch = AgenticRAGOrchestrator(company_ticker=company_ticker, filing_type=filing_type)

    # Execute query
    result = orch.query(question, verbose=verbose)

    # Display formatted response
    print(orch.format_response(result))

    return result

# Example usage
# result = query_sec_edgar(
#     "What are the main business risks mentioned in the filing?",
#     company_ticker="AAPL",
#     filing_type="10-K",
#     verbose=True
# )


## 7. Advanced Features

### Batch Processing Multiple Questions


In [ ]:
def batch_query(questions: List[str], company_ticker: str = "AAPL", filing_type: str = "10-K") -> List[Dict[str, Any]]:
    """Process multiple questions in batch."""
    orchestrator = AgenticRAGOrchestrator(company_ticker=company_ticker, filing_type=filing_type)
    results = []

    for i, question in enumerate(questions, 1):
        print(f"\n{'='*80}")
        print(f"Processing Question {i}/{len(questions)}")
        print(f"{'='*80}")
        result = orchestrator.query(question, verbose=False)
        results.append(result)
        print(f"\nAnswer: {result['answer'][:200]}...")

    return results

# Example batch processing
# questions = [
#     "What are the main business risks?",
#     "What is the revenue breakdown by segment?",
#     "What are the key financial metrics?"
# ]
# batch_results = batch_query(questions, company_ticker="AAPL")


### Export Results


In [ ]:
def export_results(result: Dict[str, Any], filename: str = "rag_result.json"):
    """Export query results to JSON file."""
    # Convert documents to serializable format
    exportable_result = result.copy()

    # Remove non-serializable document objects
    if "sources" in exportable_result:
        for source in exportable_result["sources"]:
            if "document" in source:
                source["document_content"] = source["document"].page_content[:500]
                source["document_metadata"] = source["document"].metadata
                del source["document"]

    with open(filename, 'w') as f:
        json.dump(exportable_result, f, indent=2, default=str)

    print(f"Results exported to {filename}")

# Example export
# export_results(result, "apple_10k_analysis.json")
